# 01. 수도권 아파트 버블 탐지 — EDA

- 대상: 서울 / 경기 / 인천 (2013.01 ~ 2025.12)
- 데이터: `data/processed/features_sido_monthly.csv` (468행 × 42컬럼)
- 목적:
  1. 데이터 기본 현황 확인
  2. bubble_label 분포 및 기준 검토
  3. 주요 피처 시계열 시각화
  4. 피처 간 상관관계
  5. 시도별 비교
  6. bubble_label 재검토

In [ ]:
import sys
sys.stdout.reconfigure(encoding='utf-8')
import os, warnings
warnings.filterwarnings('ignore')
os.chdir('c:/Users/bko05/Desktop/seoul-bubble-detection')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import Patch

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

ENC = 'utf-8-sig'
df = pd.read_csv('data/processed/features_sido_monthly.csv', encoding=ENC)
df['ym_dt'] = pd.to_datetime(df['ym'].astype(str), format='%Y%m')
df['year']  = df['ym'] // 100

SIDO_COLORS  = {'서울': '#E63946', '경기': '#2A9D8F', '인천': '#E9C46A'}

print(f'shape: {df.shape}')
print(f'기간 : {df.ym.min()} ~ {df.ym.max()}')
print(f'시도 : {sorted(df["시도"].unique())}')
print(f'컬럼 ({len(df.columns)}):', df.columns.tolist())

---
## 1. 데이터 기본 현황

In [ ]:
pd.set_option('display.float_format', '{:.2f}'.format)
df.drop(columns=['ym','ym_dt','year','시도','검색트렌드_유효','bubble_label']).describe().T

In [ ]:
nulls = df.isnull().sum()
nulls = nulls[nulls > 0].reset_index()
nulls.columns = ['컬럼', '결측수']
nulls['결측률(%)'] = (nulls['결측수'] / len(df) * 100).round(1)
nulls['비고'] = '초기 12개월 YoY 계산 불가 (정상)'
print(nulls.to_string(index=False))

---
## 2. Bubble Label 분포 및 기준 검토

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 전체 분포
lbl_map = {0.0: '정상(0)', 1.0: '과열(1)', 2.0: '버블(2)'}
colors   = ['#ADB5BD', '#F4A261', '#E63946']
vals = [int(df['bubble_label'].eq(k).sum()) for k in [0.0, 1.0, 2.0]]
bars = axes[0].bar(list(lbl_map.values()), vals, color=colors, edgecolor='white', linewidth=1.2)
for bar, v in zip(bars, vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, str(v),
                 ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('전체 bubble_label 분포 (유효 432행)', fontsize=12)
axes[0].set_ylabel('행 수')

# 시도별 누적 막대
sido_list = ['서울', '경기', '인천']
for i, sido in enumerate(sido_list):
    sub  = df[df['시도']==sido]
    bottom = 0
    for lbl, color in zip([0.0, 1.0, 2.0], colors):
        v = int(sub['bubble_label'].eq(lbl).sum())
        axes[1].bar(sido, v, bottom=bottom, color=color,
                    label=lbl_map[lbl] if i==0 else '')
        if v > 0:
            axes[1].text(i, bottom+v/2, str(v), ha='center', va='center',
                         fontsize=10, fontweight='bold', color='white')
        bottom += v
axes[1].set_title('시도별 bubble_label 분포', fontsize=12)
axes[1].legend()

plt.tight_layout()
plt.savefig('notebooks/fig_01_label_dist.png', bbox_inches='tight')
plt.show()

In [ ]:
print('=== 버블(2) 구간 ===')
print(df[df['bubble_label']==2][['ym','시도','매매중위_YoY','전세가율','PIR']].to_string(index=False))
print()
print('=== 2020~2021 서울 (알려진 버블 시기) ===')
peak = df[(df['시도']=='서울')&(df['ym']>=202001)&(df['ym']<=202112)]
print(peak[['ym','매매중위_YoY','전세가율','PIR','bubble_label']].to_string(index=False))

In [ ]:
# 전세가율 vs 매매YoY 산점도 — 확정 레이블 경계선
fig, ax = plt.subplots(figsize=(10, 6))
valid = df[df['bubble_label'].notna()].copy()

for lbl, color, name, marker, sz in [
    (0,'#ADB5BD','정상','o',30),(1,'#F4A261','과열','s',60),(2,'#E63946','버블','*',120)]:
    sub = valid[valid['bubble_label']==lbl]
    ax.scatter(sub['전세가율'], sub['매매중위_YoY'], c=color,
               label=f'{name}({len(sub)})', alpha=0.6, s=sz,
               marker=marker, edgecolors='white', linewidth=0.5)

# 확정 기준 (2025.05): 버블=전세가율<55 AND YoY>12, 과열=전세가율<65 AND YoY>6
ax.axhline(12, color='#E63946', ls='--', lw=1.2, alpha=0.8, label='버블기준 YoY>12%')
ax.axhline(6,  color='#F4A261', ls='--', lw=1.2, alpha=0.8, label='과열기준 YoY>6%')
ax.axvline(55, color='#E63946', ls=':',  lw=1.2, alpha=0.8, label='버블기준 전세가율<55%')
ax.axvline(65, color='#F4A261', ls=':',  lw=1.2, alpha=0.8, label='과열기준 전세가율<65%')

ax.set_xlabel('전세가율 (%)', fontsize=12)
ax.set_ylabel('매매중위가격 YoY (%)', fontsize=12)
ax.set_title('전세가율 vs 매매YoY — 확정 레이블 기준 (전세가율<55 & YoY>12 = 버블)', fontsize=13)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('notebooks/fig_02_label_scatter.png', bbox_inches='tight')
plt.show()

---
## 3. 주요 피처 시계열

In [ ]:
# 매매중위가격 + 버블구간 하이라이트
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
for ax, sido in zip(axes, ['서울','경기','인천']):
    sub = df[df['시도']==sido].sort_values('ym_dt')
    c = SIDO_COLORS[sido]
    ax.plot(sub['ym_dt'], sub['매매중위가격'], color=c, lw=2)
    ax.fill_between(sub['ym_dt'], sub['매매중위가격'], alpha=0.12, color=c)
    for _, row in sub[sub['bubble_label']==2].iterrows():
        ax.axvspan(row['ym_dt']-pd.Timedelta(days=15),
                   row['ym_dt']+pd.Timedelta(days=15), alpha=0.35, color='#E63946', zorder=0)
    for _, row in sub[sub['bubble_label']==1].iterrows():
        ax.axvspan(row['ym_dt']-pd.Timedelta(days=15),
                   row['ym_dt']+pd.Timedelta(days=15), alpha=0.15, color='#F4A261', zorder=0)
    ax.set_title(f'{sido} 매매중위가격', fontsize=12)
    ax.set_ylabel('만원')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
    ax.legend(handles=[Patch(color=c,label='가격'),
                        Patch(color='#E63946',alpha=0.4,label='버블'),
                        Patch(color='#F4A261',alpha=0.3,label='과열')], fontsize=9, loc='upper left')
    ax.grid(alpha=0.3)
plt.suptitle('시도별 아파트 매매 중위가격 (2013~2025)', fontsize=14)
plt.tight_layout()
plt.savefig('notebooks/fig_03_price_timeseries.png', bbox_inches='tight')
plt.show()

In [ ]:
# 전세가율 + PIR
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
for sido in ['서울','경기','인천']:
    sub = df[df['시도']==sido].sort_values('ym_dt')
    axes[0].plot(sub['ym_dt'], sub['전세가율'], color=SIDO_COLORS[sido], lw=1.8, label=sido)
    axes[1].plot(sub['ym_dt'], sub['PIR'],      color=SIDO_COLORS[sido], lw=1.8, label=sido)

axes[0].axhline(60, color='#E63946', ls='--', lw=1, alpha=0.7, label='60%')
axes[0].axhline(50, color='#E63946', ls=':',  lw=1, alpha=0.7, label='50%')
axes[0].set_title('전세가율 (%)', fontsize=12)
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

axes[1].axhline(10, color='#F4A261', ls='--', lw=1, alpha=0.7, label='PIR=10')
axes[1].set_title('PIR (매매중위가격 / 연간가구소득)', fontsize=12)
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('notebooks/fig_04_jeonse_pir.png', bbox_inches='tight')
plt.show()

In [ ]:
# 거시지표
macro = df[df['시도']=='서울'].sort_values('ym_dt')
fig, axes = plt.subplots(3, 2, figsize=(15, 10))
plots = [
    ('base_rate','기준금리 (%)','#264653'),
    ('mortgage_rate','주담대금리 (%)','#2A9D8F'),
    ('m2_yoy_pct','M2 YoY (%)','#E76F51'),
    ('cpi_yoy_pct','CPI YoY (%)','#F4A261'),
    ('bsi_realestate','부동산BSI','#E9C46A'),
    ('가계대출비중','서울 가계대출비중 (%)','#E63946'),
]
for ax, (col, title, color) in zip(axes.flatten(), plots):
    ax.plot(macro['ym_dt'], macro[col], color=color, lw=1.8)
    ax.fill_between(macro['ym_dt'], macro[col], alpha=0.12, color=color)
    ax.set_title(title, fontsize=11)
    ax.grid(alpha=0.3)
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('거시/금융 지표 (2013~2025)', fontsize=14)
plt.tight_layout()
plt.savefig('notebooks/fig_05_macro.png', bbox_inches='tight')
plt.show()

In [ ]:
# 공급지표
fig, axes = plt.subplots(3, 1, figsize=(15, 9), sharex=True)
for ax, (col, title) in zip(axes, [
    ('인허가','주택 인허가 (호/월)'),
    ('준공','주택 준공 (호/월)'),
    ('미분양','공사완료후 미분양 (호)'),
]):
    for sido in ['서울','경기','인천']:
        sub = df[df['시도']==sido].sort_values('ym_dt')
        ax.plot(sub['ym_dt'], sub[col], color=SIDO_COLORS[sido], lw=1.5, label=sido)
    ax.set_title(title, fontsize=12)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.suptitle('공급 지표 (2013~2025)', fontsize=14)
plt.tight_layout()
plt.savefig('notebooks/fig_06_supply.png', bbox_inches='tight')
plt.show()

---
## 4. 피처 간 상관관계

In [ ]:
corr_cols = [
    '매매중위가격','매매중위_YoY','전세가율','PIR',
    'base_rate','mortgage_rate','rate_spread',
    'm2_yoy_pct','cpi_yoy_pct','bsi_realestate',
    '미분양','인허가_yoy','차주당대출_십만원','가계대출비중',
    '아파트매매_검색',
]
seoul = df[df['시도']=='서울'][corr_cols].dropna()
corr  = seoul.corr()

fig, ax = plt.subplots(figsize=(13, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', vmin=-1, vmax=1, center=0,
            linewidths=0.5, ax=ax, annot_kws={'size':8})
ax.set_title('서울 피처 상관관계 히트맵', fontsize=13)
plt.tight_layout()
plt.savefig('notebooks/fig_07_corr_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
from scipy.stats import pointbiserialr

valid = df[df['bubble_label'].notna() & (df['시도']=='서울')].copy()
y = valid['bubble_label'].astype(float)

res = []
for col in corr_cols:
    x = valid[col]; mask = x.notna()
    if mask.sum() < 20: continue
    r, p = pointbiserialr(y[mask], x[mask])
    res.append({'피처': col, '상관계수': round(r,3), 'p값': round(p,4)})

corr_df = pd.DataFrame(res).sort_values('상관계수', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 7))
colors_bar = ['#E63946' if v>0 else '#2A9D8F' for v in corr_df['상관계수']]
ax.barh(corr_df['피처'], corr_df['상관계수'], color=colors_bar, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_title('bubble_label과 각 피처 상관관계 (서울)', fontsize=12)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('notebooks/fig_08_feature_corr.png', bbox_inches='tight')
plt.show()
print(corr_df.to_string(index=False))

---
## 5. 시도별 PIR 추이 + 최신 현황

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for sido in ['서울','경기','인천']:
    sub = df[df['시도']==sido].groupby('year')['PIR'].mean().reset_index()
    ax.plot(sub['year'], sub['PIR'], color=SIDO_COLORS[sido],
            marker='o', ms=5, lw=2, label=sido)
ax.axhline(10, color='gray', ls='--', lw=1, alpha=0.7, label='PIR=10')
ax.set_title('시도별 PIR 연도별 추이', fontsize=13)
ax.set_xlabel('연도'); ax.set_ylabel('PIR (배)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('notebooks/fig_09_pir_trend.png', bbox_inches='tight')
plt.show()

print('\n=== 2025.12 최신 현황 ===')
latest = df[df['ym']==202512][['시도','매매중위가격','PIR','전세가율','미분양','가계대출비중']].copy()
latest.columns = ['시도','매매중위(만원)','PIR','전세가율(%)','미분양(호)','가계대출비중(%)']
print(latest.to_string(index=False))

---
## 6. Bubble Label 최종 확정 기준

**확정 (2025.05)**:
- 버블(2): 전세가율 < **55%** AND 매매YoY > **12%**
- 과열(1): 전세가율 < **65%** AND 매매YoY > **6%**
- 정상(0): 그 외 | NaN: 초기 12개월 (YoY 계산 불가)

이전 기준 (전세가율<50, YoY>15) 대비 변경 이유:
- 이전: 버블 5건 → 불균형비율 78:1, LSTM 학습 불가
- 확정: 버블 27건 → 불균형비율 13.7:1, 2020~2021 서울 급등기 포착

In [ ]:
# 확정 기준 bubble_label 분포 상세
print('=== 확정 기준 bubble_label 분포 ===')
print('기준: 버블=전세가율<55 & YoY>12 / 과열=전세가율<65 & YoY>6')
print()
print('전체:')
print(df['bubble_label'].value_counts(dropna=False).sort_index())
print()
print('시도별:')
for sido in ['서울','경기','인천']:
    sub = df[df['시도']==sido]
    b = int(sub['bubble_label'].eq(2).sum())
    o = int(sub['bubble_label'].eq(1).sum())
    n = int(sub['bubble_label'].eq(0).sum())
    na = int(sub['bubble_label'].isna().sum())
    print(f'  {sido}: 버블(2)={b} 과열(1)={o} 정상(0)={n} NaN={na}')
print()
print('=== 버블(2) 구간 상세 ===')
b2 = df[df['bubble_label']==2][['ym','시도','매매중위_YoY','전세가율','PIR']]
print(b2.to_string(index=False))

In [ ]:
# 확정 기준 버블 구간 가격 시계열 시각화
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
for ax, sido in zip(axes, ['서울','경기','인천']):
    sub = df[df['시도']==sido].sort_values('ym_dt')
    c = SIDO_COLORS[sido]
    ax.plot(sub['ym_dt'], sub['매매중위가격'], color=c, lw=2)
    ax.fill_between(sub['ym_dt'], sub['매매중위가격'], alpha=0.1, color=c)
    for _, row in sub[sub['bubble_label']==2].iterrows():
        ax.axvspan(row['ym_dt']-pd.Timedelta(days=15),
                   row['ym_dt']+pd.Timedelta(days=15), alpha=0.35, color='#E63946', zorder=0)
    for _, row in sub[sub['bubble_label']==1].iterrows():
        ax.axvspan(row['ym_dt']-pd.Timedelta(days=15),
                   row['ym_dt']+pd.Timedelta(days=15), alpha=0.13, color='#F4A261', zorder=0)
    ax.set_title(f'{sido} 매매중위가격 — 버블 구간 하이라이트 (확정기준)', fontsize=11)
    ax.set_ylabel('만원')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
    ax.legend(handles=[Patch(color=c,label='가격'),
                        Patch(color='#E63946',alpha=0.4,label=f'버블(2)={int(sub.bubble_label.eq(2).sum())}건'),
                        Patch(color='#F4A261',alpha=0.3,label=f'과열(1)={int(sub.bubble_label.eq(1).sum())}건')],
              fontsize=9, loc='upper left')
    ax.grid(alpha=0.3)
plt.suptitle('시도별 버블 구간 (확정 기준: 전세가율<55% & YoY>12%)', fontsize=14)
plt.tight_layout()
plt.savefig('notebooks/fig_10_bubble_highlight.png', bbox_inches='tight')
plt.show()

---
## 7. EDA 요약

In [ ]:
print('=' * 55)
print('  EDA 요약')
print('=' * 55)

print('\n[데이터]')
print('  468행 × 42컬럼 (3시도 × 156개월, 2013.01~2025.12)')
print('  실질 결측: YoY 초기 12개월만 → 정상')

print('\n[가격 (2025.12 기준)]')
for sido in ['서울','경기','인천']:
    sub  = df[df['시도']==sido]
    pnow = sub[sub['ym']==202512]['매매중위가격'].values[0]
    pmax = sub['매매중위가격'].max()
    print(f'  {sido}: 현재={pnow:,.0f}만원, 최고={pmax:,.0f}만원')

print('\n[PIR (2025.12)]')
for sido in ['서울','경기','인천']:
    pir = df[(df['시도']==sido)&(df['ym']==202512)]['PIR'].values[0]
    print(f'  {sido}: {pir:.1f}배 → 연소득 {pir:.1f}년치')

print('\n[Bubble Label 확정 기준]')
print('  버블(2): 전세가율<55% AND YoY>12%')
print('  과열(1): 전세가율<65% AND YoY>6%')
print(f'  버블(2): {int(df["bubble_label"].eq(2).sum())}건 (서울 2020~2021 + 2023~2025)')
print(f'  과열(1): {int(df["bubble_label"].eq(1).sum())}건')
print(f'  정상(0): {int(df["bubble_label"].eq(0).sum())}건')
print(f'  불균형비율: {int(df["bubble_label"].eq(0).sum())}:{int(df["bubble_label"].eq(2).sum())} = {int(df["bubble_label"].eq(0).sum())/max(1,int(df["bubble_label"].eq(2).sum())):.1f}:1')

print('\n[다음 단계]')
print('  1. 피처 정규화 (MinMaxScaler)')
print('  2. LSTM-Autoencoder 모델링 (notebooks/03_lstm_autoencoder.ipynb)')
print('  3. HMM 레짐 감지 (notebooks/04_hmm_regime.ipynb)')
print('  4. Granger 공간 분석 (notebooks/05_granger_spatial.ipynb)')
print('  5. CUSUM 변환점 탐지 (notebooks/06_cusum.ipynb)')
print('  6. 앙상블 + SHAP (notebooks/07_ensemble_shap_heatmap.ipynb)')
print('=' * 55)